# FinGuard Fraud Detection Pipeline
## Notebook 03 — Gold Layer Feature Engineering

Produces business-ready, consumption-optimised tables: per-transaction fraud
signals, aggregated customer risk scores (Delta MERGE upsert), and a flat
ML feature store.

Source: `finguard.silver.transactions_enriched`
Target: `finguard.gold.*`

## Imports

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

print(f"Spark version   : {spark.version}")
print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Configuration

In [0]:
CATALOG_NAME  = "finguard"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

SILVER_TRANSACTIONS_ENRICHED = f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions_enriched"
SILVER_CUSTOMERS             = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers"

GOLD_FRAUD_FEATURES      = f"{CATALOG_NAME}.{GOLD_SCHEMA}.fraud_features"
GOLD_CUSTOMER_RISK_SCORES = f"{CATALOG_NAME}.{GOLD_SCHEMA}.customer_risk_scores"
GOLD_ML_FEATURE_STORE    = f"{CATALOG_NAME}.{GOLD_SCHEMA}.ml_feature_store"

AUSTRAC_THRESHOLD       = 10_000.00
HIGH_VELOCITY_THRESHOLD = 5       # transactions within 10 minutes
HIGH_AMOUNT_MULTIPLIER  = 3.0     # amount > 3x customer 30d average = anomaly

BATCH_ID      = f"gold_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PIPELINE_NAME = "finguard_gold_feature_engineering"

print(f"Batch ID : {BATCH_ID}")
print(f"Source   : {CATALOG_NAME}.{SILVER_SCHEMA}.*")
print(f"Target   : {CATALOG_NAME}.{GOLD_SCHEMA}.*")

## Create Gold Schema

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}
    COMMENT 'FinGuard Gold layer — business-ready fraud feature tables'
""")

print(f"✓ Schema ready: {CATALOG_NAME}.{GOLD_SCHEMA}")

## Read Silver Data

In [0]:
silver_df = spark.table(SILVER_TRANSACTIONS_ENRICHED)

total = silver_df.count()
print(f"Silver transactions loaded : {total:,}")
print(f"Columns available          : {len(silver_df.columns)}")

required_cols = [
    "transaction_id", "customer_id", "amount", "is_fraud",
    "spend_7d_aud", "spend_30d_aud", "txn_count_7d",
    "is_rapid_succession", "is_near_austrac_threshold",
    "merchant_risk_level", "is_high_risk", "amount_vs_30d_avg",
]
missing = [c for c in required_cols if c not in silver_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
print("✓ All required columns present")

## Fraud Feature Engineering

A rule-based scorecard: ten binary signals, each contributing one point to
a composite `risk_score` (0–10), banded into `risk_band`.

In [0]:
fraud_features_df = (
    silver_df
    .withColumn("flag_amount_anomaly",
        F.col("amount_vs_30d_avg") > HIGH_AMOUNT_MULTIPLIER)
    .withColumn("flag_velocity_attack",
        F.col("is_rapid_succession") & (F.col("txn_count_7d") > HIGH_VELOCITY_THRESHOLD))
    .withColumn("flag_structuring",
        F.col("is_near_austrac_threshold"))
    .withColumn("flag_high_risk_merchant",
        F.col("merchant_risk_level") == "high")
    .withColumn("flag_night_high_value",
        F.col("is_night") & (F.col("amount") > 500))
    .withColumn("flag_international",
        F.col("merchant_is_international"))
    .withColumn("flag_kyc_unverified",
        ~F.col("kyc_verified") & (F.col("amount") > 1_000))
    .withColumn("flag_first_transaction",
        F.col("is_first_transaction"))
    .withColumn("flag_card_not_present",
        F.col("channel") == "card_not_present")
    .withColumn("flag_high_risk_customer",
        F.col("is_high_risk"))

    .withColumn(
        "risk_score",
        F.col("flag_amount_anomaly").cast(IntegerType()) +
        F.col("flag_velocity_attack").cast(IntegerType()) +
        F.col("flag_structuring").cast(IntegerType()) +
        F.col("flag_high_risk_merchant").cast(IntegerType()) +
        F.col("flag_night_high_value").cast(IntegerType()) +
        F.col("flag_international").cast(IntegerType()) +
        F.col("flag_kyc_unverified").cast(IntegerType()) +
        F.col("flag_first_transaction").cast(IntegerType()) +
        F.col("flag_card_not_present").cast(IntegerType()) +
        F.col("flag_high_risk_customer").cast(IntegerType())
    )
    .withColumn(
        "risk_band",
        F.when(F.col("risk_score") == 0, F.lit("no_risk"))
         .when(F.col("risk_score") <= 2,  F.lit("low"))
         .when(F.col("risk_score") <= 4,  F.lit("medium"))
         .when(F.col("risk_score") <= 6,  F.lit("high"))
         .otherwise(F.lit("critical"))
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .withColumn("_batch_id",          F.lit(BATCH_ID))
    .withColumn("_pipeline_name",     F.lit(PIPELINE_NAME))
)

print(f"✓ Fraud features engineered")
print(f"  Total columns: {len(fraud_features_df.columns)}")

In [0]:
print("Risk band distribution:")
display(
    fraud_features_df
    .groupBy("risk_band")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.round(F.avg("amount"), 2).alias("avg_amount_aud"),
        F.sum(F.col("is_fraud").cast(IntegerType())).alias("actual_fraud_count"),
        F.round(
            F.sum(F.col("is_fraud").cast(IntegerType())) /
            F.count("transaction_id") * 100, 2
        ).alias("fraud_rate_pct")
    )
    .orderBy(F.desc("transaction_count"))
)

## Write Gold Fraud Features Table

In [0]:
(
    fraud_features_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("risk_band")
    .saveAsTable(GOLD_FRAUD_FEATURES)
)

count = spark.table(GOLD_FRAUD_FEATURES).count()
print(f"✓ {GOLD_FRAUD_FEATURES}: {count:,} rows")

spark.sql(f"""
    OPTIMIZE {GOLD_FRAUD_FEATURES}
    ZORDER BY (customer_id, txn_date)
""")
print("✓ OPTIMIZE complete")

## Customer Risk Scores via Delta MERGE

Aggregates all transactions per customer into a single risk profile row.
First run creates the table; subsequent runs MERGE (upsert) so only
customers with new transactions are updated.

In [0]:
customer_risk_df = (
    fraud_features_df
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.round(F.sum("amount"), 2).alias("total_spend_aud"),
        F.round(F.avg("amount"), 2).alias("avg_transaction_aud"),
        F.round(F.max("amount"), 2).alias("max_transaction_aud"),

        F.sum(F.col("is_fraud").cast(IntegerType())).alias("total_fraud_count"),
        F.round(
            F.sum(F.col("is_fraud").cast(IntegerType())) /
            F.count("transaction_id") * 100, 4
        ).alias("fraud_rate_pct"),

        F.sum(F.col("flag_amount_anomaly").cast(IntegerType())).alias("count_amount_anomaly"),
        F.sum(F.col("flag_velocity_attack").cast(IntegerType())).alias("count_velocity_attack"),
        F.sum(F.col("flag_structuring").cast(IntegerType())).alias("count_structuring"),
        F.sum(F.col("flag_high_risk_merchant").cast(IntegerType())).alias("count_high_risk_merchant"),
        F.sum(F.col("flag_night_high_value").cast(IntegerType())).alias("count_night_high_value"),
        F.sum(F.col("flag_international").cast(IntegerType())).alias("count_international"),
        F.sum(F.col("flag_card_not_present").cast(IntegerType())).alias("count_card_not_present"),

        F.round(F.avg("risk_score"), 4).alias("avg_risk_score"),
        F.max("risk_score").alias("max_risk_score"),

        F.max("txn_timestamp").alias("last_transaction_at"),
        F.min("txn_timestamp").alias("first_transaction_at"),
        F.max("risk_band").alias("highest_risk_band"),
    )
    .withColumn(
        "customer_risk_category",
        F.when(F.col("fraud_rate_pct") > 10,   F.lit("critical"))
         .when(F.col("fraud_rate_pct") > 5,    F.lit("high"))
         .when(F.col("fraud_rate_pct") > 2,    F.lit("medium"))
         .when(F.col("total_fraud_count") > 0, F.lit("low"))
         .otherwise(F.lit("clean"))
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .withColumn("_batch_id",          F.lit(BATCH_ID))
    .withColumn("_pipeline_name",     F.lit(PIPELINE_NAME))
)

print(f"✓ Customer risk scores computed: {customer_risk_df.count():,} customers")

In [0]:
table_exists = spark.catalog.tableExists(GOLD_CUSTOMER_RISK_SCORES)

if not table_exists:
    print("First run — creating customer risk scores table...")
    (
        customer_risk_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_CUSTOMER_RISK_SCORES)
    )
    print(f"✓ Table created: {GOLD_CUSTOMER_RISK_SCORES}")

else:
    print("Subsequent run — merging customer risk scores...")
    target_table = DeltaTable.forName(spark, GOLD_CUSTOMER_RISK_SCORES)
    (
        target_table.alias("target")
        .merge(customer_risk_df.alias("source"), F.expr("target.customer_id = source.customer_id"))
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✓ MERGE complete: {GOLD_CUSTOMER_RISK_SCORES}")

count = spark.table(GOLD_CUSTOMER_RISK_SCORES).count()
print(f"  Total customers: {count:,}")

## Delta Transaction History

Every MERGE, INSERT, or OPTIMIZE operation is recorded in the Delta transaction
log. DESCRIBE HISTORY exposes this as an auditable record of every write — who
ran it, what operation, how many rows were affected — useful for debugging and
for satisfying audit/lineage requirements in a regulated environment.

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {GOLD_CUSTOMER_RISK_SCORES}")
    .select("version", "timestamp", "operation", "operationMetrics")
    .orderBy(F.desc("version"))
    .limit(5)
)

## ML Feature Store

A flat, ML-ready feature table: one row per transaction, label = `is_fraud`,
no PII. This is what a Data Scientist would feed into a training pipeline.

In [0]:
customer_risk_slim = spark.table(GOLD_CUSTOMER_RISK_SCORES).select(
    F.col("customer_id"),
    F.col("avg_risk_score").alias("customer_avg_risk_score"),
    F.col("fraud_rate_pct").alias("customer_historical_fraud_rate"),
    F.col("customer_risk_category"),
    F.col("total_transactions").alias("customer_total_transactions"),
    F.col("count_velocity_attack").alias("customer_velocity_count"),
    F.col("count_international").alias("customer_intl_count"),
)

ml_feature_df = (
    fraud_features_df
    .join(F.broadcast(customer_risk_slim), on="customer_id", how="left")
    .select(
        F.col("transaction_id"),
        F.col("customer_id"),
        F.col("txn_date"),

        F.col("amount"),
        F.col("txn_hour"),
        F.col("is_weekend").cast(IntegerType()).alias("is_weekend"),
        F.col("is_night").cast(IntegerType()).alias("is_night"),
        F.col("amount_band"),

        (F.col("channel") == "card_not_present").cast(IntegerType()).alias("is_cnp"),
        (F.col("channel") == "tap_to_pay").cast(IntegerType()).alias("is_tap"),
        (F.col("channel") == "osko").cast(IntegerType()).alias("is_osko"),
        F.col("merchant_is_international").cast(IntegerType()).alias("is_international"),
        F.col("merchant_is_online").cast(IntegerType()).alias("is_online_merchant"),

        F.col("spend_7d_aud"),
        F.col("spend_30d_aud"),
        F.col("txn_count_7d"),
        F.col("txn_count_30d"),
        F.col("avg_amount_7d"),
        F.col("amount_vs_30d_avg"),
        F.col("seconds_since_prev_txn"),
        F.col("is_rapid_succession").cast(IntegerType()).alias("is_rapid_succession"),
        F.col("customer_txn_sequence"),
        F.col("is_first_transaction").cast(IntegerType()).alias("is_first_transaction"),

        F.col("flag_amount_anomaly").cast(IntegerType()).alias("flag_amount_anomaly"),
        F.col("flag_velocity_attack").cast(IntegerType()).alias("flag_velocity_attack"),
        F.col("flag_structuring").cast(IntegerType()).alias("flag_structuring"),
        F.col("flag_high_risk_merchant").cast(IntegerType()).alias("flag_high_risk_merchant"),
        F.col("flag_night_high_value").cast(IntegerType()).alias("flag_night_high_value"),
        F.col("flag_international").cast(IntegerType()).alias("flag_international"),
        F.col("flag_kyc_unverified").cast(IntegerType()).alias("flag_kyc_unverified"),
        F.col("flag_card_not_present").cast(IntegerType()).alias("flag_card_not_present"),
        F.col("flag_high_risk_customer").cast(IntegerType()).alias("flag_high_risk_customer"),
        F.col("risk_score"),
        F.col("risk_band"),

        F.col("credit_score"),
        F.col("age_years"),
        F.col("income_band"),
        F.col("account_tenure_days"),
        F.col("employment_status"),
        F.col("kyc_verified").cast(IntegerType()).alias("kyc_verified"),

        F.col("customer_avg_risk_score"),
        F.col("customer_historical_fraud_rate"),
        F.col("customer_risk_category"),
        F.col("customer_total_transactions"),
        F.col("customer_velocity_count"),
        F.col("customer_intl_count"),

        F.col("is_fraud").cast(IntegerType()).alias("label"),

        F.col("_gold_processed_at"),
        F.col("_batch_id"),
    )
)

print(f"✓ ML feature store built")
print(f"  Total rows    : {ml_feature_df.count():,}")
print(f"  Total features: {len(ml_feature_df.columns) - 4}")  # minus IDs and audit cols

In [0]:
(
    ml_feature_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("risk_band")
    .saveAsTable(GOLD_ML_FEATURE_STORE)
)

count = spark.table(GOLD_ML_FEATURE_STORE).count()
print(f"✓ {GOLD_ML_FEATURE_STORE}: {count:,} rows")

spark.sql(f"""
    OPTIMIZE {GOLD_ML_FEATURE_STORE}
    ZORDER BY (customer_id, txn_date)
""")
print("✓ OPTIMIZE complete")

## Gold Layer Data Quality

In [0]:
fraud_df   = spark.table(GOLD_FRAUD_FEATURES)
risk_df    = spark.table(GOLD_CUSTOMER_RISK_SCORES)
ml_df      = spark.table(GOLD_ML_FEATURE_STORE)

fraud_total   = fraud_df.count()
risk_total    = risk_df.count()
ml_total      = ml_df.count()

print("═" * 60)
print("  Gold Layer — Data Quality Report")
print("═" * 60)
print(f"  fraud_features       : {fraud_total:>10,} rows")
print(f"  customer_risk_scores : {risk_total:>10,} rows")
print(f"  ml_feature_store     : {ml_total:>10,} rows")

print("  Risk band distribution (fraud_features):")
risk_dist = (
    fraud_df.groupBy("risk_band")
    .agg(
        F.count("*").alias("count"),
        F.round(F.sum(F.col("is_fraud").cast(IntegerType())) /
                F.count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .orderBy(F.desc("count"))
    .collect()
)
for row in risk_dist:
    print(f"    {row['risk_band']:<12} {row['count']:>10,} rows  fraud rate: {row['fraud_rate_pct']}%")

print("\n  Customer risk categories:")
cat_dist = risk_df.groupBy("customer_risk_category").count().orderBy(F.desc("count")).collect()
for row in cat_dist:
    print(f"    {row['customer_risk_category']:<12} {row['count']:>8,} customers")

fraud_label = ml_df.filter(F.col("label") == 1).count()
clean_label = ml_df.filter(F.col("label") == 0).count()
print(f"\n  ML label balance:")
print(f"    Fraud (label=1) : {fraud_label:>10,}  ({fraud_label/ml_total*100:.2f}%)")
print(f"    Clean (label=0) : {clean_label:>10,}  ({clean_label/ml_total*100:.2f}%)")
print("═" * 60)

## Full Pipeline Summary

In [0]:
print("═" * 65)
print("  FINGUARD PIPELINE — COMPLETE")
print("  All 3 layers running end to end")
print("═" * 65)

print("  BRONZE (finguard.bronze.*)")
for t in ["transactions", "customers", "merchants"]:
    c = spark.table(f"finguard.bronze.{t}").count()
    print(f"    finguard.bronze.{t:<25} {c:>10,}")

print("  SILVER (finguard.silver.*)")
for t in ["transactions_cleaned", "transactions_enriched", "customers"]:
    c = spark.table(f"finguard.silver.{t}").count()
    print(f"    finguard.silver.{t:<25} {c:>10,}")

print("  GOLD (finguard.gold.*)")
for t in ["fraud_features", "customer_risk_scores", "ml_feature_store"]:
    c = spark.table(f"finguard.gold.{t}").count()
    print(f"    finguard.gold.{t:<27} {c:>10,}")

print("  Features engineered : 10 fraud signals + composite risk score")
print("  Delta MERGE         : customer_risk_scores (incremental upsert)")
print("  ML feature store    : label + 35 features, privacy-safe (no PII)")
print()
print("  Next → workflows/finguard_pipeline_workflow.yml")
print("         (Databricks Workflow to orchestrate all 3 notebooks)")
print("═" * 65)